# EP10 — Logic Neural Networks & Soft Logic
**COMPSCI 713 · S1 2025 Q1 · 2 marks**

| Part | Topic | Marks |
|------|-------|-------|
| a | Compute LNN bounds + classify truth value | 1 |
| b | Draw/describe syntax tree of formula | 1 |

Key rule: `((Explosive(x) ∧ Ignited(x)) ∨ ToxicGas(x)) → Hazardous(x)`

---

## What is a Logic Neural Network (LNN)?

A **Logic Neural Network** bridges symbolic logic and neural networks:
- Each logical connective (AND, OR, →) is a **neuron**
- Instead of crisp True/False, each formula has a **confidence interval [L, U]**
  - **L** = lower bound (how confident we are it's true)
  - **U** = upper bound (upper limit on truth confidence)
- A formula is **Definitely True** if `L ≥ α`, **Definitely False** if `U < α`, otherwise **Uncertain**

Threshold α = 0.7 in this question.

---

## Soft Logic Operations

| Operation | Formula |
|-----------|----------|
| A ∧ B | `A × B` |
| A ∨ B | `A + B − A × B` |

For bounds:
- **AND lower bound:** `L_A × L_B`
- **AND upper bound:** `U_A × U_B`
- **OR lower bound:** `L_A + L_B − L_A × L_B`
- **OR upper bound:** `U_A + U_B − U_A × U_B`

---

## Part (a) — Worked Solution [1 mark]

Given bounds:
- Explosive(Z1): L=0.9, U=1.0
- Ignited(Z1):   L=0.0, U=0.1
- ToxicGas(Z1):  L=0.6, U=0.8
- Threshold α = 0.7

**Step 1 — Compute (Explosive ∧ Ignited):**

```
L_AND = L_Explosive × L_Ignited = 0.9 × 0.0 = 0.00
U_AND = U_Explosive × U_Ignited = 1.0 × 0.1 = 0.10
```

**Step 2 — Compute ((Explosive ∧ Ignited) ∨ ToxicGas):**

Let A = (Explosive ∧ Ignited), B = ToxicGas

```
L_OR = L_A + L_B − L_A × L_B
     = 0.00 + 0.6 − (0.00 × 0.6)
     = 0.60

U_OR = U_A + U_B − U_A × U_B
     = 0.10 + 0.8 − (0.10 × 0.8)
     = 0.90 − 0.08
     = 0.82
```

Result: **[0.60, 0.82]**

**Step 3 — Classify:**
- L = 0.60 < α = 0.70 → not Definitely True
- U = 0.82 ≥ α = 0.70 → not Definitely False
- **Classification: UNCERTAIN** (interval straddles α)

---

## Part (b) — Syntax Tree [1 mark]

The formula `((Explosive ∧ Ignited) ∨ ToxicGas) → Hazardous` as a tree:

```
            →
           / \
          ∨   Hazardous
         / \
        ∧   ToxicGas
       / \
Explosive  Ignited
```

- Root node: `→` (implication neuron)
- Left child of `→`: `∨` (OR neuron)
  - Left child of `∨`: `∧` (AND neuron)
    - Leaves: `Explosive`, `Ignited`
  - Right child of `∨`: `ToxicGas`
- Right child of `→`: `Hazardous`

Each internal node = a neuron in the LNN. Leaf nodes = input predicates.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# =============================================================
# Soft logic calculator — verifies Part (a) step by step
# =============================================================

def soft_and(a_l, a_u, b_l, b_u):
    """Soft AND: A × B (product t-norm)"""
    return a_l * b_l, a_u * b_u

def soft_or(a_l, a_u, b_l, b_u):
    """Soft OR: A + B − A×B (product t-conorm)"""
    return (a_l + b_l - a_l * b_l,
            a_u + b_u - a_u * b_u)

def classify(l, u, alpha=0.7):
    if l >= alpha:  return 'Definitely True  ✅'
    if u < alpha:   return 'Definitely False ❌'
    return 'Uncertain ⚠️'

# --- Given values ---
exp_l, exp_u  = 0.9, 1.0   # Explosive(Z1)
ign_l, ign_u  = 0.0, 0.1   # Ignited(Z1)
tox_l, tox_u  = 0.6, 0.8   # ToxicGas(Z1)
alpha         = 0.7

# Step 1: AND
and_l, and_u = soft_and(exp_l, exp_u, ign_l, ign_u)

# Step 2: OR
or_l, or_u = soft_or(and_l, and_u, tox_l, tox_u)

print('=== Soft Logic Computation (LNN) ===')
print(f'Explosive ∧ Ignited:  [{and_l:.2f}, {and_u:.2f}]')
print(f'  └─ {and_l:.2f} = {exp_l} × {ign_l}  (lower)')
print(f'  └─ {and_u:.2f} = {exp_u} × {ign_u}  (upper)')
print()
print(f'(Exp∧Ign) ∨ ToxicGas: [{or_l:.2f}, {or_u:.2f}]')
print(f'  └─ {or_l:.2f} = {and_l} + {tox_l} − {and_l}×{tox_l}  (lower)')
print(f'  └─ {or_u:.2f} = {and_u} + {tox_u} − {and_u}×{tox_u}  (upper)')
print()
print(f'Threshold α = {alpha}')
print(f'L = {or_l:.2f} {"≥" if or_l >= alpha else "<"} α  → {"contributes to True" if or_l >= alpha else "not definitely true"}')
print(f'U = {or_u:.2f} {"≥" if or_u >= alpha else "<"} α  → {"could be true" if or_u >= alpha else "definitely false"}')
print(f'Classification: {classify(or_l, or_u, alpha)}')

In [ ]:
# =============================================================
# Visualisation 1: Confidence interval on the [0,1] scale
# =============================================================
fig, ax = plt.subplots(figsize=(10, 3.5))

intervals = [
    ('Explosive(Z1)',       0.9, 1.0, '#2ecc71'),
    ('Ignited(Z1)',         0.0, 0.1, '#e74c3c'),
    ('ToxicGas(Z1)',        0.6, 0.8, '#f39c12'),
    ('Exp ∧ Ign',          and_l, and_u, '#9b59b6'),
    ('(Exp∧Ign) ∨ ToxGas', or_l, or_u, '#3498db'),
]

for i, (label, l, u, colour) in enumerate(reversed(intervals)):
    y = i
    ax.barh(y, u - l, left=l, height=0.5, color=colour, alpha=0.8, edgecolor='white')
    ax.text(u + 0.01, y, f'[{l:.2f}, {u:.2f}]', va='center', fontsize=9)
    ax.text(-0.01, y, label, ha='right', va='center', fontsize=9)

# Threshold line
ax.axvline(alpha, color='black', linestyle='--', linewidth=1.5, label=f'α = {alpha}')
ax.text(alpha + 0.01, len(intervals) - 0.3, f'α={alpha}', fontsize=9)

ax.set_xlim(-0.3, 1.15)
ax.set_ylim(-0.5, len(intervals) - 0.2)
ax.set_xlabel('Confidence [0, 1]')
ax.set_yticks([])
ax.set_title('LNN Confidence Intervals\nInterval spanning α = Uncertain; all-above = True; all-below = False')
ax.spines[['left','top','right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================
# Visualisation 2: Syntax tree (LNN architecture)
# =============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

fig, ax = plt.subplots(figsize=(9, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_title('LNN Syntax Tree: ((Explosive ∧ Ignited) ∨ ToxicGas) → Hazardous',
             fontsize=11, fontweight='bold')

# Node positions: (x, y, label, colour)
nodes = {
    'implies':    (5.0, 5.5, '→',           '#e74c3c'),
    'or':         (3.0, 4.0, '∨',           '#e67e22'),
    'hazardous':  (7.0, 4.0, 'Hazardous',   '#27ae60'),
    'and':        (1.8, 2.5, '∧',           '#3498db'),
    'toxic':      (4.2, 2.5, 'ToxicGas',    '#9b59b6'),
    'explosive':  (0.8, 1.0, 'Explosive',   '#7f8c8d'),
    'ignited':    (2.8, 1.0, 'Ignited',     '#7f8c8d'),
}

# Draw edges
edges = [
    ('implies', 'or'),
    ('implies', 'hazardous'),
    ('or', 'and'),
    ('or', 'toxic'),
    ('and', 'explosive'),
    ('and', 'ignited'),
]
for src, dst in edges:
    x0, y0 = nodes[src][0], nodes[src][1]
    x1, y1 = nodes[dst][0], nodes[dst][1]
    ax.annotate('', xy=(x1, y1 + 0.3), xytext=(x0, y0 - 0.3),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

# Draw nodes
for key, (x, y, label, colour) in nodes.items():
    circle = plt.Circle((x, y), 0.35, color=colour, zorder=5)
    ax.add_patch(circle)
    ax.text(x, y, label if len(label) <= 3 else label[:4]+'.',
            ha='center', va='center', fontsize=8,
            color='white', fontweight='bold', zorder=6)
    # Full label below leaf nodes
    if key in ('explosive', 'ignited', 'toxic', 'hazardous'):
        ax.text(x, y - 0.5, label, ha='center', va='top', fontsize=7.5, color='#333')

# Legend
for colour, label in [('#e74c3c','→ implication'), ('#e67e22','∨ OR'),
                       ('#3498db','∧ AND'), ('#9b59b6','∨ leaf'), ('#27ae60','output'),
                       ('#7f8c8d','input predicates')]:
    ax.add_patch(mpatches.Patch(color=colour, label=label))

ax.legend(loc='lower right', fontsize=8, framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================
# Interactive: what happens to classification as bounds change?
# Shows the Uncertain / True / False regions across α
# =============================================================
alphas = np.linspace(0, 1, 200)
results = []
for a in alphas:
    if or_l >= a:
        results.append(2)   # Definitely True
    elif or_u < a:
        results.append(0)   # Definitely False
    else:
        results.append(1)   # Uncertain

fig, ax = plt.subplots(figsize=(9, 2.5))
colours = {0: '#e74c3c', 1: '#f39c12', 2: '#2ecc71'}
labels  = {0: 'Definitely False', 1: 'Uncertain', 2: 'Definitely True'}

prev = None
for i, (a, r) in enumerate(zip(alphas, results)):
    if r != prev:
        start = a
    ax.axvspan(a, a + 1/200, color=colours[r], alpha=0.6)
    prev = r

ax.axvline(alpha, color='black', linewidth=2, linestyle='--', label=f'Exam α={alpha}')
ax.axvline(or_l, color='blue', linewidth=1.5, linestyle=':', label=f'L={or_l:.2f}')
ax.axvline(or_u, color='purple', linewidth=1.5, linestyle=':', label=f'U={or_u:.2f}')

for val, label in [(0,'False'), (1,'Uncertain'), (2,'True')]:
    ax.add_patch(mpatches.Patch(color=colours[val], alpha=0.6, label=labels[val]))

ax.set_xlabel('Threshold α')
ax.set_yticks([])
ax.set_title(f'Truth classification of (Exp∧Ign)∨ToxGas = [{or_l:.2f},{or_u:.2f}] vs threshold α')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()

print(f'\nAt exam threshold α={alpha}:')
print(f'  Interval [{or_l:.2f}, {or_u:.2f}] → {classify(or_l, or_u, alpha)}')

## Exam Quick-Reference

**Soft logic formula:**
- A ∧ B = A × B  (lower: L_A × L_B, upper: U_A × U_B)
- A ∨ B = A + B − A×B  (apply to L and U separately)

**Classification rule (threshold α):**
- L ≥ α → **Definitely True**
- U < α → **Definitely False**
- Otherwise → **Uncertain**

**Syntax tree structure** — root is the outermost connective, leaves are predicates. Each node = one LNN neuron.

**This question's answer:**
- (Exp∧Ign): [0.00, 0.10]
- (Exp∧Ign)∨ToxGas: [0.60, 0.82]
- At α=0.7: **Uncertain** (L < α but U > α)